# BC wildfire-smoke benchmark (2 years) on Colab

Trains AirLapseV2 and the baselines on datasets 8-10 (Oct 2024 - Sep 2026,
southern BC + northern WA PurpleAir, hourly; 8 = no fire, 9 = distance-only
fire, 10 = wind-transported fire). Everything persistent lives in Google
Drive under `DRIVE_DIR`, so a disconnected runtime loses nothing.

**Runtime → Change runtime type → GPU** before running.

The data files are never in the GitHub repo (PurpleAir's terms bar
redistribution). Either put them in `DRIVE_DIR/data/` yourself (step 3, option A),
or build them here (option B).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bcsmoke'   # data/, cache_long/, results/ live here
BRANCH = 'feat/sensor-dataset-4'

import os
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)

## 1. Get the code

In [ ]:
%cd /content
!rm -rf PM25GNN && git clone -q -b {BRANCH} https://github.com/Grace-VN/PM25GNN.git
%cd /content/PM25GNN
!pip install -q -r requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## 2. Data

**Option A (recommended):** build on your own machine with
`python pilot/build_bcsmoke_long.py`, then upload `data/BCSmoke2y_*.npy`,
`data/BCSmoke2y_monthly.csv` and `data/site_bcsmoke2y.txt` to `DRIVE_DIR/data/`.
Open-Meteo rate-limits per IP and Colab IPs are shared, so building at home is
more predictable.

**Option B:** set `BUILD_HERE = True`. Add Colab secrets (key icon in the left
sidebar) `PURPLEAIR_API_KEY` and `FIRMS_MAP_KEY` (free:
https://firms.modaps.eosdis.nasa.gov/api/map_key/). Takes a few hours, mostly
throttling under the free weather-API limits; the cache is on Drive, so if it
stops (disconnect, daily limit) just run the cell again.

In [ ]:
BUILD_HERE = False

if BUILD_HERE:
    from google.colab import userdata
    os.environ['PURPLEAIR_API_KEY'] = userdata.get('PURPLEAIR_API_KEY')
    os.environ['FIRMS_MAP_KEY'] = userdata.get('FIRMS_MAP_KEY')
    !python -u pilot/build_bcsmoke_long.py --cache "{DRIVE_DIR}/cache_long" --out "{DRIVE_DIR}/data"

In [ ]:
# copy the dataset files from Drive into the repo's data/ (where config.yaml points)
!cp -v "{DRIVE_DIR}"/data/BCSmoke2y_* "{DRIVE_DIR}"/data/site_bcsmoke2y.txt data/
import pandas as pd
monthly = pd.read_csv('data/BCSmoke2y_monthly.csv', index_col=0)
monthly   # check Jul 2026 (val) and Aug-Sep 2026 (test) actually contain smoke

## 3. Benchmark

Results go to `DRIVE_DIR/results/baseline_bench_ds<N>.csv`. Re-running a cell
only retrains the models you list (their old rows are replaced), so after a
disconnect just re-run with the missing models.

Start with 1 repeat to see timings; train.py's output is captured, so the cell
is quiet until each model finishes.

In [ ]:
os.environ['PILOT_RESULTS_DIR'] = f'{DRIVE_DIR}/results'
MODELS = 'AirLapseV2 LSTM GRU PM25_GNN Informer PatchTST'
!python pilot/baseline_bench.py --dataset 10 --models {MODELS} --epochs 50 --repeats 1 --early_stop 10

## 4. Does the fire signal help? (AirLapseV2 + the strongest baseline on 8 / 9 / 10)

Then raise `--repeats` to 3-5 for the numbers that go in a paper.

In [ ]:
for ds in (8, 9):
    !python pilot/baseline_bench.py --dataset {ds} --models AirLapseV2 GRU --epochs 50 --repeats 1 --early_stop 10

In [ ]:
import glob
for fp in sorted(glob.glob(f'{DRIVE_DIR}/results/baseline_bench_ds*.csv')):
    df = pd.read_csv(fp)
    df = df[df.status == 'ok']
    print(os.path.basename(fp))
    display(df.groupby('model')[['RMSE', 'RMSE_smoke', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']]
              .mean().round(3).sort_values('RMSE'))